# Day 02 — Attention Mechanism

**Time budget: 60 minutes.** Segment headings carry their own timebox. If you overrun badly, stop
and split the topic across two days rather than rushing the hands-on parts.

## How to use this notebook

1. Read the markdown, then run the code cell under it before reading on. The cells build on each
   other, so run them in order.
2. When a cell says *predict first*, write down your guess before running it. Being wrong is the
   part that sticks.
3. Finish with the exercises at the bottom. Solutions are there, but try first.

## Agenda

| # | Segment | Time |
| - | ------- | ---- |
| 0 | The sequential bottleneck | 3 min |
| 1 | RNN memory failure | 8 min |
| 2 | Attention: looking everywhere | 12 min |
| 3 | Build attention from scratch | 10 min |
| 4 | Transformer attention context | 5 min |
| 5 | Self-attention deep dive | 15 min |
| 6 | Multi-head attention | 7 min |
| 7 | Exercises and quiz | — |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print("Ready to explore attention mechanisms!")

---
## 0. The Sequential Bottleneck (3 min)

Why do we need attention? The problem is **sequential processing**. Traditional RNNs read text word by word, left to right. But human understanding doesn't work that way.

Consider this sentence: "The cat that the dog chased ran away."

To understand "ran," you need to remember "cat" from 7 words ago. RNNs struggle with this because each step overwrites the previous hidden state. By the time we reach "ran," the information about "cat" might be gone.

**The key insight:** What if we could look at ALL words at once when processing ANY word?

In [ ]:
# Let's visualize the problem with a simple example
sentence = ["The", "cat", "that", "the", "dog", "chased", "ran", "away"]
print(f"Sentence: {' '.join(sentence)}")
print(f"Length: {len(sentence)} words")
print("\nRNN processing order:")
for i, word in enumerate(sentence):
    print(f"Step {i+1}: Processing '{word}' - can only see: {sentence[:i+1]}")

---
## 1. RNN Memory Failure (8 min)

Let's build a simple RNN and watch it forget. We'll create a task where understanding requires remembering the first word to interpret the last word.

In [ ]:
# Simple RNN implementation
class SimpleRNN:
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        # Small random weights
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.1
        
    def forward(self, inputs):
        batch_size, seq_len, input_size = inputs.shape
        h = np.zeros((batch_size, self.hidden_size))
        hidden_states = []
        
        for t in range(seq_len):
            h = np.tanh(inputs[:, t] @ self.W_xh.T + h @ self.W_hh.T)
            hidden_states.append(h.copy())
            
        return np.array(hidden_states)

# Test with dummy data
rnn = SimpleRNN(input_size=10, hidden_size=8)
print(f"RNN created with {rnn.hidden_size} hidden units")

In [ ]:
# Create a sequence where we need to remember the first element
def create_memory_task(seq_length=20, batch_size=1):
    """Create sequences where first element determines correct answer at the end"""
    sequences = np.random.randn(batch_size, seq_length, 10)
    # First element is special - set it to a clear pattern
    sequences[:, 0, :] = 5.0  # Strong signal at the start
    # Middle elements are noise
    sequences[:, 1:-1, :] = np.random.randn(batch_size, seq_length-2, 10) * 0.1
    return sequences

# Test sequence
test_seq = create_memory_task(seq_length=15)
print(f"Input shape: {test_seq.shape}")
print(f"First element (signal): {test_seq[0, 0, :3]}... (should be strong)")
print(f"Middle element (noise): {test_seq[0, 7, :3]}... (should be weak)")

In [ ]:
# Run RNN and see how the memory fades
hidden_states = rnn.forward(test_seq)

# Measure how much the hidden state changes from the initial strong signal
initial_state = hidden_states[0, 0]  # After processing first element
state_similarities = []

for t in range(len(hidden_states)):
    current_state = hidden_states[t, 0]
    # Cosine similarity to measure how much we've "forgotten"
    similarity = np.dot(initial_state, current_state) / (np.linalg.norm(initial_state) * np.linalg.norm(current_state))
    state_similarities.append(similarity)

# Plot the forgetting curve
plt.figure(figsize=(10, 4))
plt.plot(state_similarities, 'r-', linewidth=2)
plt.xlabel('Time Step')
plt.ylabel('Similarity to Initial State')
plt.title('RNN Memory Decay: How Much We Remember the First Word')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
plt.show()

print(f"Initial similarity: {state_similarities[0]:.3f}")
print(f"Final similarity: {state_similarities[-1]:.3f}")
print(f"Memory retention: {state_similarities[-1]/state_similarities[0]*100:.1f}%")

**The failure is clear:** The RNN's hidden state drifts away from its initial configuration. By the end of the sequence, it has "forgotten" most information about the beginning.

This is the **vanishing gradient problem** in action, and it's why RNNs struggle with long-range dependencies.

---
## 2. Attention: Looking Everywhere (12 min)

Attention solves this by asking: **"What if we don't have to remember everything? What if we can just look back at what we need, when we need it?"**

The core insight:
1. Store all previous states (no forgetting!)
2. When processing a new word, compute how relevant each previous word is
3. Take a weighted average of previous states based on relevance

This is **attention**: selectively focusing on relevant parts of the input.

In [ ]:
def simple_attention(query, keys, values):
    """
    Simple attention mechanism
    query: what we're looking for (current position)
    keys: what's available to look at (all previous positions) 
    values: the actual information at each position
    """
    # Compute similarity scores (how relevant each key is to our query)
    scores = np.dot(keys, query)  # Simple dot product similarity
    
    # Convert to attention weights (probabilities)
    attention_weights = softmax(scores)
    
    # Weighted average of values
    attended_output = np.sum(attention_weights[:, None] * values, axis=0)
    
    return attended_output, attention_weights

def softmax(x):
    """Numerically stable softmax"""
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

print("Attention function ready!")

In [ ]:
# Let's test attention on a simple example
# Imagine we're processing "The cat ran" and we're at "ran"
# We want to find what "ran" refers to

# Create simple word embeddings (normally these would be learned)
embeddings = {
    'the': np.array([0.1, 0.2, 0.1]),
    'cat': np.array([0.8, 0.1, 0.7]),  # animal-like features
    'ran': np.array([0.2, 0.9, 0.1])   # action-like features
}

words = ['the', 'cat', 'ran']
sequence_embeddings = np.array([embeddings[w] for w in words])

print("Word embeddings:")
for i, word in enumerate(words):
    print(f"{word}: {sequence_embeddings[i]}")

# When processing "ran", we want to know which previous word is most relevant
query = embeddings['ran']  # What we're looking for
keys = sequence_embeddings[:2]  # Previous words: "the", "cat"
values = sequence_embeddings[:2]  # Same as keys in this simple case

attended_output, attention_weights = simple_attention(query, keys, values)

print("\nAttention analysis for 'ran':")
for i, (word, weight) in enumerate(zip(words[:2], attention_weights)):
    print(f"Attention to '{word}': {weight:.3f} ({weight*100:.1f}%)")

print(f"\nAttended output: {attended_output}")
print(f"This is closest to: {words[np.argmax([np.dot(attended_output, emb) for emb in sequence_embeddings[:2]])]}")

In [ ]:
# Visualize attention weights
plt.figure(figsize=(8, 3))
bars = plt.bar(words[:2], attention_weights, color=['lightblue', 'orange'])
plt.ylabel('Attention Weight')
plt.title('Attention Weights: Which word does "ran" focus on?')
plt.ylim(0, 1)

# Add value labels on bars
for bar, weight in zip(bars, attention_weights):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{weight:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nInterpretation: 'ran' pays {attention_weights[1]*100:.1f}% attention to 'cat'")
print("This makes sense - actions typically focus on the actor (the noun)!")

**Key insight:** Attention learns to focus on relevant parts of the input. The verb "ran" correctly identifies "cat" as the most relevant previous word, not "the".

This is much more powerful than RNN's sequential memory because:
1. **No forgetting**: All previous states are preserved
2. **Selective access**: We only use what's relevant
3. **Learned relevance**: The model learns what to pay attention to

---
## 3. Build Attention from Scratch (10 min)

Now let's implement a proper attention mechanism from scratch. We'll build the version used in transformers: **scaled dot-product attention**.

In [ ]:
class AttentionLayer:
    def __init__(self, d_model):
        self.d_model = d_model
        # Transform input into Query, Key, Value representations
        self.W_q = np.random.randn(d_model, d_model) * 0.1
        self.W_k = np.random.randn(d_model, d_model) * 0.1  
        self.W_v = np.random.randn(d_model, d_model) * 0.1
        
    def forward(self, x):
        """
        x: input embeddings [seq_len, d_model]
        Returns: attention output [seq_len, d_model]
        """
        seq_len, d_model = x.shape
        
        # Generate Query, Key, Value matrices
        Q = x @ self.W_q  # [seq_len, d_model]
        K = x @ self.W_k  # [seq_len, d_model] 
        V = x @ self.W_v  # [seq_len, d_model]
        
        # Compute attention scores: Q * K^T
        scores = Q @ K.T  # [seq_len, seq_len]
        
        # Scale by sqrt(d_model) to prevent extremely large values
        scores = scores / np.sqrt(d_model)
        
        # Apply softmax to get attention weights
        attention_weights = self.softmax_2d(scores)
        
        # Apply attention to values
        output = attention_weights @ V  # [seq_len, d_model]
        
        return output, attention_weights
    
    def softmax_2d(self, x):
        """Apply softmax along the last dimension"""
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# Test our attention layer
attention = AttentionLayer(d_model=4)
print(f"Attention layer created with d_model={attention.d_model}")

In [ ]:
# Create a test sequence
test_sequence = np.array([
    [1.0, 0.5, 0.2, 0.1],  # Token 1
    [0.2, 1.0, 0.3, 0.4],  # Token 2  
    [0.1, 0.3, 1.0, 0.2],  # Token 3
    [0.4, 0.2, 0.1, 1.0]   # Token 4
])

print("Input sequence:")
print(test_sequence)
print(f"Shape: {test_sequence.shape}")

# Run attention
output, weights = attention.forward(test_sequence)

print("\nAttention output:")
print(output)
print(f"Shape: {output.shape}")

In [ ]:
# Visualize the attention pattern
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(weights, cmap='Blues', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Weights Matrix')
plt.xticks(range(4), [f'Pos {i}' for i in range(4)])
plt.yticks(range(4), [f'Pos {i}' for i in range(4)])

# Add text annotations
for i in range(4):
    for j in range(4):
        plt.text(j, i, f'{weights[i,j]:.2f}', 
                ha='center', va='center', 
                color='white' if weights[i,j] > 0.5 else 'black')

plt.subplot(1, 2, 2)
# Show how much each position attends to itself vs others
self_attention = np.diag(weights)
other_attention = 1 - self_attention

x = range(4)
plt.bar(x, self_attention, label='Self-attention', alpha=0.7)
plt.bar(x, other_attention, bottom=self_attention, label='Other-attention', alpha=0.7)
plt.xlabel('Position')
plt.ylabel('Attention Weight')
plt.title('Self vs Other Attention')
plt.legend()
plt.xticks(x, [f'Pos {i}' for i in x])

plt.tight_layout()
plt.show()

print("\nAttention pattern analysis:")
for i in range(4):
    most_attended = np.argmax(weights[i])
    print(f"Position {i} attends most to position {most_attended} (weight: {weights[i, most_attended]:.3f})")

**What we've built:** A complete attention mechanism! Each position can now "look at" every other position and decide how much to focus on each one.

Notice:
- Each row sums to 1 (probability distribution)
- Positions can attend to themselves or to other positions
- The pattern emerges from the learned Q, K, V transformations

---
## 4. Transformer Attention Context (5 min)

Let's put our attention mechanism in context. In a real transformer:
- **Sequence length**: 512-8192 tokens (GPT-3 uses 2048, GPT-4 uses up to 32k)
- **Model dimension**: 768-1536 for base models, up to 12,288 for large models  
- **Attention heads**: 12-96 heads running in parallel
- **Computational cost**: O(n²) in sequence length

In [ ]:
# Calculate real-world attention costs
def attention_cost(seq_len, d_model, num_heads):
    """Calculate memory and computation for attention"""
    # Attention matrix is [seq_len, seq_len] per head
    attention_memory = seq_len * seq_len * num_heads
    
    # Matrix multiplications: Q@K^T and Attn@V
    flops = 2 * seq_len * seq_len * d_model * num_heads
    
    return attention_memory, flops

# GPT-3 scale
models = [
    ("GPT-2 Small", 512, 768, 12),
    ("GPT-3 Base", 2048, 768, 12), 
    ("GPT-3 Large", 2048, 1536, 24),
    ("GPT-4 (estimated)", 8192, 1536, 48)
]

print("Real-world attention costs:")
print(f"{'Model':<20} {'Seq Len':<8} {'Attention Matrix':<15} {'GFLOPs':<10}")
print("-" * 60)

for name, seq_len, d_model, heads in models:
    memory, flops = attention_cost(seq_len, d_model, heads)
    matrix_size = f"{seq_len}x{seq_len}"
    gflops = flops / 1e9
    print(f"{name:<20} {seq_len:<8} {matrix_size:<15} {gflops:<10.1f}")

print("\nKey insight: Attention cost grows quadratically with sequence length!")
print("This is why we need techniques like sparse attention for very long sequences.")

---
## 5. Self-Attention Deep Dive (15 min)

Now let's understand the **Query, Key, Value** paradigm more deeply. This is the heart of transformer attention.

**Intuition:**
- **Query**: "What am I looking for?"
- **Key**: "What do I offer?"
- **Value**: "What information do I actually contain?"

Think of it like a library search:
- You have a **query** ("I need information about cats")
- Books have **keys** (their topics/keywords)
- Books contain **values** (the actual content)
- You find books whose keys match your query, then read their values

In [ ]:
# Let's build a more interpretable example
class InterpretableAttention:
    def __init__(self):
        # We'll set the weights manually to see what happens
        pass
    
    def manual_attention(self, embeddings, q_transform, k_transform, v_transform):
        """
        Apply attention with manually set transformations
        """
        # Transform embeddings into Q, K, V
        Q = embeddings @ q_transform
        K = embeddings @ k_transform  
        V = embeddings @ v_transform
        
        # Compute attention
        scores = Q @ K.T
        weights = self.softmax_2d(scores)
        output = weights @ V
        
        return output, weights, Q, K, V
    
    def softmax_2d(self, x):
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# Create interpretable word embeddings
# Dimension 0: "nouniness", Dimension 1: "verbiness", Dimension 2: "importance"
word_embeddings = np.array([
    [0.9, 0.1, 0.5],  # "cat" - high noun, low verb, medium importance
    [0.1, 0.9, 0.8],  # "chased" - low noun, high verb, high importance  
    [0.8, 0.1, 0.3],  # "mouse" - high noun, low verb, low importance
    [0.1, 0.8, 0.6]   # "quickly" - low noun, medium verb, medium importance
])

words = ["cat", "chased", "mouse", "quickly"]

print("Word embeddings (noun, verb, importance):")
for i, word in enumerate(words):
    print(f"{word:<8}: {word_embeddings[i]}")

attention = InterpretableAttention()

In [ ]:
# Design specific attention patterns

# Pattern 1: Verbs should attend to nouns (subject-verb relationship)
# Q: look for verbs, K: offer nouns, V: provide noun information
verb_to_noun_q = np.array([
    [0.0, 1.0, 0.0],  # Query verbs (high verbiness)
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0]
])

verb_to_noun_k = np.array([
    [1.0, 0.0, 0.0],  # Offer nouns (high nouniness)
    [0.0, 0.0, 0.0], 
    [0.0, 0.0, 0.0]
])

verb_to_noun_v = np.eye(3)  # Return original embedding

output1, weights1, Q1, K1, V1 = attention.manual_attention(
    word_embeddings, verb_to_noun_q, verb_to_noun_k, verb_to_noun_v
)

print("\nVerb-to-Noun Attention Pattern:")
print("Attention weights (who attends to whom):")
print(f"{'':>10}", end="")
for word in words:
    print(f"{word:>8}", end="")
print()

for i, word in enumerate(words):
    print(f"{word:>10}", end="")
    for j in range(len(words)):
        print(f"{weights1[i,j]:>8.2f}", end="")
    print()

# Analyze the pattern
print("\nAnalysis:")
for i, word in enumerate(words):
    most_attended = np.argmax(weights1[i])
    if weights1[i, most_attended] > 0.3:  # Significant attention
        print(f"'{word}' attends strongly to '{words[most_attended]}' ({weights1[i, most_attended]:.2f})")

In [ ]:
# Pattern 2: Importance-based attention (attend to important words)
importance_q = np.array([
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 1.0]  # Query for importance
])

importance_k = np.array([
    [0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0], 
    [0.0, 0.0, 1.0]  # Offer importance
])

importance_v = np.eye(3)

output2, weights2, Q2, K2, V2 = attention.manual_attention(
    word_embeddings, importance_q, importance_k, importance_v
)

print("\nImportance-based Attention Pattern:")
print("Words ranked by how much attention they receive:")

# Sum attention received by each word
attention_received = np.sum(weights2, axis=0)
word_attention = [(words[i], attention_received[i]) for i in range(len(words))]
word_attention.sort(key=lambda x: x[1], reverse=True)

for word, attn in word_attention:
    original_importance = word_embeddings[words.index(word), 2]
    print(f"{word:<8}: receives {attn:.2f} attention (original importance: {original_importance:.1f})")

In [ ]:
# Visualize both attention patterns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Verb-to-noun attention
im1 = axes[0].imshow(weights1, cmap='Reds', aspect='auto')
axes[0].set_title('Verb→Noun Attention')
axes[0].set_xlabel('Keys (what we can attend to)')
axes[0].set_ylabel('Queries (who is attending)')
axes[0].set_xticks(range(len(words)))
axes[0].set_yticks(range(len(words)))
axes[0].set_xticklabels(words)
axes[0].set_yticklabels(words)
plt.colorbar(im1, ax=axes[0])

# Add annotations
for i in range(len(words)):
    for j in range(len(words)):
        text = axes[0].text(j, i, f'{weights1[i,j]:.2f}',
                           ha="center", va="center", 
                           color="white" if weights1[i,j] > 0.5 else "black")

# Plot 2: Importance attention  
im2 = axes[1].imshow(weights2, cmap='Blues', aspect='auto')
axes[1].set_title('Importance-based Attention')
axes[1].set_xlabel('Keys (what we can attend to)')
axes[1].set_ylabel('Queries (who is attending)')
axes[1].set_xticks(range(len(words)))
axes[1].set_yticks(range(len(words)))
axes[1].set_xticklabels(words)
axes[1].set_yticklabels(words)
plt.colorbar(im2, ax=axes[1])

for i in range(len(words)):
    for j in range(len(words)):
        text = axes[1].text(j, i, f'{weights2[i,j]:.2f}',
                           ha="center", va="center",
                           color="white" if weights2[i,j] > 0.5 else "black")

plt.tight_layout()
plt.show()

print("\nKey insight: Different Q/K/V transformations create different attention patterns!")
print("This is how transformers learn different types of relationships.")

**This demonstrates the power of Q/K/V:**
- The **same input** can produce **different attention patterns** depending on the Q/K/V transformations
- Each pattern captures a different type of relationship (syntactic, semantic, importance-based)
- In real transformers, these transformations are **learned** to capture useful patterns for the task

---
## 6. Multi-Head Attention (7 min)

**The final piece:** Why do we need multiple attention heads?

A single attention head can only learn one type of relationship. But language has many types:
- Syntactic (subject-verb, adjective-noun)
- Semantic (word meanings, coreference)
- Positional (nearby words, distant dependencies)
- Task-specific (sentiment, factual relationships)

**Solution:** Run multiple attention heads in parallel, each learning different patterns!

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Separate Q, K, V for each head
        self.heads = []
        for _ in range(num_heads):
            head = {
                'W_q': np.random.randn(d_model, self.d_k) * 0.1,
                'W_k': np.random.randn(d_model, self.d_k) * 0.1,
                'W_v': np.random.randn(d_model, self.d_k) * 0.1
            }
            self.heads.append(head)
        
        # Output projection
        self.W_o = np.random.randn(d_model, d_model) * 0.1
    
    def attention_head(self, x, head_weights):
        """Single attention head"""
        Q = x @ head_weights['W_q']
        K = x @ head_weights['W_k'] 
        V = x @ head_weights['W_v']
        
        scores = Q @ K.T / np.sqrt(self.d_k)
        weights = self.softmax_2d(scores)
        output = weights @ V
        
        return output, weights
    
    def forward(self, x):
        """Multi-head attention"""
        head_outputs = []
        head_weights_list = []
        
        # Run each head
        for head in self.heads:
            output, weights = self.attention_head(x, head)
            head_outputs.append(output)
            head_weights_list.append(weights)
        
        # Concatenate head outputs
        concatenated = np.concatenate(head_outputs, axis=1)
        
        # Final linear projection
        final_output = concatenated @ self.W_o
        
        return final_output, head_weights_list
    
    def softmax_2d(self, x):
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# Create multi-head attention
mha = MultiHeadAttention(d_model=6, num_heads=3)
print(f"Created multi-head attention: {mha.num_heads} heads, {mha.d_k} dims per head")

In [ ]:
# Test with a longer sequence
test_seq = np.random.randn(6, 6)  # 6 tokens, 6-dimensional embeddings
tokens = ["The", "quick", "brown", "fox", "jumps", "over"]

output, head_weights = mha.forward(test_seq)

print(f"Input shape: {test_seq.shape}")
print(f"Output shape: {output.shape}")
print(f"Number of attention heads: {len(head_weights)}")
print(f"Each head attention shape: {head_weights[0].shape}")

In [ ]:
# Visualize what each head learned
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for head_idx in range(3):
    weights = head_weights[head_idx]
    
    im = axes[head_idx].imshow(weights, cmap='viridis', aspect='auto')
    axes[head_idx].set_title(f'Head {head_idx + 1} Attention Pattern')
    axes[head_idx].set_xlabel('Key Position')
    axes[head_idx].set_ylabel('Query Position')
    axes[head_idx].set_xticks(range(len(tokens)))
    axes[head_idx].set_yticks(range(len(tokens)))
    axes[head_idx].set_xticklabels(tokens, rotation=45)
    axes[head_idx].set_yticklabels(tokens)
    
    plt.colorbar(im, ax=axes[head_idx])
    
    # Add text annotations for strong connections
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            if weights[i, j] > 0.3:  # Only show strong connections
                axes[head_idx].text(j, i, f'{weights[i,j]:.2f}',
                                   ha="center", va="center", 
                                   color="white", fontsize=8)

plt.tight_layout()
plt.show()

print("\nMulti-head analysis:")
for head_idx in range(3):
    weights = head_weights[head_idx]
    # Find the strongest attention pattern for this head
    max_pos = np.unravel_index(np.argmax(weights), weights.shape)
    strength = weights[max_pos]
    print(f"Head {head_idx + 1}: Strongest connection from '{tokens[max_pos[0]]}' to '{tokens[max_pos[1]]}' ({strength:.3f})")

print("\nKey insight: Each head can specialize in different types of relationships!")
print("In real transformers, heads learn patterns like:")
print("- Head 1: Subject-verb relationships")
print("- Head 2: Adjective-noun relationships")
print("- Head 3: Long-range dependencies")
print("- Head 4: Positional relationships")

**Multi-head attention summary:**
1. **Parallel specialization**: Each head learns different relationship types
2. **Richer representations**: Multiple perspectives on the same input
3. **Ensemble effect**: Heads can complement each other's weaknesses
4. **Interpretability**: We can analyze what each head focuses on

This is why transformers are so powerful - they can simultaneously track syntax, semantics, and task-specific relationships!

---
## 7. Exercises

Try each before opening the solution.

**Exercise 1.** Implement masked attention for autoregressive language modeling. Modify the attention mechanism so that position i can only attend to positions 0 through i (not future positions).

**Exercise 2.** Create an attention visualization for the sentence "The cat that the dog chased ran" where you manually design Q/K/V matrices to make "ran" attend strongly to "cat".

**Exercise 3.** Compare attention vs RNN memory: Create a sequence where important information is at position 0, noise in the middle, and a query at the end. Show that attention preserves the information while RNN loses it.

**Exercise 4.** Implement positional attention: Design an attention head that makes each word attend primarily to words at specific relative positions (e.g., +1, -1, +2).

**Exercise 5.** Analyze attention complexity: For a sequence of length N, calculate how the memory and computation requirements scale. Compare to RNN which is O(N).

In [ ]:
# Your scratch space for the exercises.

<details>
<summary><b>Solutions</b> (click to expand)</summary>

```python
# Exercise 1: Masked attention
def masked_attention(Q, K, V):
    seq_len = Q.shape[0]
    scores = Q @ K.T / np.sqrt(Q.shape[1])
    
    # Create causal mask (upper triangular matrix)
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    scores = scores - mask * 1e9  # Set future positions to -inf
    
    weights = softmax_2d(scores)
    output = weights @ V
    return output, weights

# Exercise 2: Manual Q/K/V for "ran" -> "cat"
# Set Q to detect action words, K to detect animals, V to return animal info
sentence = ["The", "cat", "that", "the", "dog", "chased", "ran"]
# Design embeddings where dim 0 = animal, dim 1 = action
# Then Q focuses on actions, K offers animals

# Exercise 3: Attention vs RNN memory preservation  
def memory_comparison():
    signal = np.array([5.0, 0, 0])  # Strong signal
    noise = np.random.randn(10, 3) * 0.1  # Weak noise
    query = np.array([1.0, 0, 0])  # Looking for signal
    
    sequence = np.vstack([signal[None], noise, query[None]])
    
    # RNN will degrade signal through 10 noise steps
    # Attention can directly connect last position to first
    
# Exercise 4: Positional attention
def positional_attention_head(seq_len, offset=1):
    # Create Q/K that make position i attend to position i+offset
    pos_encoding = np.eye(seq_len)
    # Shift encoding by offset to create positional preference
    
# Exercise 5: Complexity analysis
def complexity_analysis(N):
    attention_memory = N * N  # O(N²)
    attention_compute = N * N * d_model  # O(N²·d)
    
    rnn_memory = N * d_model  # O(N·d) 
    rnn_compute = N * d_model * d_model  # O(N·d²)
    
    # Attention becomes prohibitive for very long sequences
```

</details>

---
## Self-check quiz

If you cannot answer these without scrolling up, reread the segment named in the answer.

1. **What is the core problem that attention solves compared to RNNs?**
2. **What do Query, Key, and Value represent conceptually?**
3. **Why do we scale attention scores by √(d_k)?**
4. **What is the computational complexity of attention in sequence length?**
5. **Why do transformers use multiple attention heads instead of just one?**

<details>
<summary><b>Answers</b></summary>

1. **RNNs process sequentially and forget earlier information due to the vanishing gradient problem. Attention allows direct connections between any two positions, eliminating the memory bottleneck.** (segment 1)

2. **Query: "What am I looking for?", Key: "What information do I offer?", Value: "What information do I actually contain?". Like a library search system.** (segment 5)

3. **To prevent attention scores from becoming extremely large, which would make the softmax too sharp and gradients vanish. The scaling keeps scores in a reasonable range.** (segment 3)

4. **O(N²) in sequence length N, because we compute attention between every pair of positions.** (segment 4)

5. **Different heads can specialize in different types of relationships (syntactic, semantic, positional). This provides multiple perspectives on the same input and richer representations.** (segment 6)

</details>

---
## Where to go next

**Next concept: Training Paradigms (Pretraining, Fine-tuning, RLHF)**

Now that you understand how transformers **process** information through attention, the next crucial question is: **how do they learn?** 

The training paradigm is what transforms a randomly initialized transformer into GPT-4. Understanding pretraining → fine-tuning → RLHF will complete your picture of how modern AI systems actually work in practice.

You've built the engine (attention), now let's see how it gets trained to be useful!